# 04 · Especialização: agente com ferramentas (item 05)
O agente funciona em duas fases:
1. **Planejamento:** o LLM recebe as ferramentas `buscar_base`, `consultar_risco` e `chuva_72h`, sem exigência de JSON, e escolhe quais chamar. A `buscar_base` sempre roda, como salvaguarda.
2. **Resposta:** os trechos e os dados do setor vão para o mesmo gerador validado do RAG.

Sem a API do produto no ar, os dados de risco do setor são **simulados** (`simulado: true`).

In [1]:
import sys, pathlib
RAIZ = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
from assistente.config import CONFIG

In [2]:
from assistente.ingest import carregar
from assistente.retriever import Buscador
from assistente.agent import Agente, TOOLS

agente = Agente(Buscador(carregar()).carregar())
print("ferramentas:", [t["function"]["name"] for t in TOOLS])

ferramentas: ['buscar_base', 'consultar_risco', 'chuva_72h']


In [3]:
def mostrar(pergunta, setor=None):
    resp, meta = agente.agente(pergunta, setor)
    print("PERGUNTA:", pergunta, "| setor:", setor)
    print("chamadas:", meta["chamadas"])
    print(resp.model_dump_json(indent=2))
    print(f"latência: {meta['latencia_s']:.1f}s\n")

## Integração com o produto: pergunta com setor
O agente consulta o risco e a chuva de 72 h do setor. Os dados vêm da API do Encosta Viva, aqui simulados.

In [4]:
mostrar("Está chovendo muito aqui, o que devo fazer?", setor="vila_sahy")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

PERGUNTA: Está chovendo muito aqui, o que devo fazer? | setor: vila_sahy
chamadas: [{'ferramenta': 'chuva_72h', 'args': {'setor_id': 'vila_sahy'}}, {'ferramenta': 'buscar_base', 'args': {'pergunta': 'Está chovendo muito aqui, o que devo fazer?'}, 'forcada': True}]
{
  "resposta": "Está chovendo muito aqui. Verifique sinais de perigo como rachaduras, trincas, inclinação, barulhos ou água barrenta. Se houver, procure um local seguro e entre em contato com a Defesa Civil por telefone 199.",
  "o_que_fazer": [
    "Em emergência ligue 199 (Defesa Civil) ou 193 (Bombeiros)."
  ],
  "fontes": [
    {
      "doc_id": "cemaden_educacao",
      "pagina": 21,
      "trecho": null
    }
  ],
  "recusa": false,
  "confianca": 1.0
}
latência: 14.3s



## Onde o agente ganha do RAG simples, e a variação entre execuções
Na pergunta abaixo (q039), os placares salvos mostram o agente acertando na v2 e na v3, e o RAG simples errando nas duas. A comparação a seguir lê esses resultados salvos, então é determinística.

In [5]:
import pandas as pd

linhas = []
for rodada, pasta in [("v2", CONFIG.dir_resultados / "v2"), ("v3", CONFIG.dir_resultados)]:
    for modo in ["rag", "agente"]:
        d = pd.read_csv(pasta / f"respostas_{modo}.csv").set_index("id")
        linhas.append({"rodada": rodada, "modo": modo, "acerto": d.loc["q039", "acerto"],
                       "resposta": d.loc["q039", "resposta"][:110]})
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(linhas)

,rodada,modo,acerto,resposta
0,v2,rag,False,A resposta gerada não pôde ser ligada a um documento oficial. Procure a Defesa Civil do seu município.
1,v2,agente,True,"Sim, a prefeitura precisa mostrar um documento que comprove que sua casa está em área de risco e que a remoção"
2,v3,rag,False,Não consegui gerar uma resposta confiável. Procure a Defesa Civil do seu município.
3,v3,agente,True,"Sim, a prefeitura precisa mostrar um documento que comprove que sua casa está em área de risco e que a remoção"


A execução ao vivo abaixo pode divergir dos placares salvos, porque o modelo de 3B varia entre execuções mesmo com temperatura 0,1. Por isso o placar (notebook 05) é a medida válida, e não um exemplo isolado. Se a resposta citar algo diferente de "vistoria, laudo técnico e notificação", ela está errada. Isso é um exemplo real da variabilidade discutida no README.

In [6]:
mostrar("Minha casa vai ser removida por estar em área de risco. A prefeitura precisa me mostrar algum documento antes?")
mostrar("Qual é o telefone da Defesa Civil?")

PERGUNTA: Minha casa vai ser removida por estar em área de risco. A prefeitura precisa me mostrar algum documento antes? | setor: None
chamadas: [{'ferramenta': 'buscar_base', 'args': {'pergunta': 'Documentos sobre remoção de casa em área de risco'}}]
{
  "resposta": "Sim, a prefeitura precisa mostrar ao menos a carta geotécnica de aptidão à urbanização.",
  "o_que_fazer": [
    "Em emergência ligue 199 (Defesa Civil) ou 193 (Bombeiros)."
  ],
  "fontes": [
    {
      "doc_id": "lei_12608_2012",
      "pagina": 1,
      "trecho": null
    }
  ],
  "recusa": false,
  "confianca": 1.0
}
latência: 9.2s



PERGUNTA: Qual é o telefone da Defesa Civil? | setor: None
chamadas: [{'ferramenta': 'buscar_base', 'args': {'pergunta': 'Qual é o telefone da Defesa Civil?'}, 'forcada': True}]
{
  "resposta": "O telefone da Defesa Civil é 199.",
  "o_que_fazer": [
    "Ligue 199 para a Defesa Civil em emergência."
  ],
  "fontes": [
    {
      "doc_id": "ppdc_santos",
      "pagina": 1,
      "trecho": null
    }
  ],
  "recusa": false,
  "confianca": 1.0
}
latência: 9.5s

